In [ ]:
import gc
import optuna
import numpy as np
import pandas as pd

from lightgbm import LGBMRanker, early_stopping, log_evaluation


DATE_COL = "dat_izdavanja"
CLIENT_COL = "klijent_id"


def make_temporal_folds(
    df,
    date_col=DATE_COL,
    val_months=3,
    min_train_months=6,
    step_months=3,
    min_train_rows=500,
    min_val_rows=100,
    reserve_last_fold=True,
):
    """
    Expanding-window temporal CV:
    Fold i:
      train = sve prije cutoff
      val   = [cutoff, cutoff + val_months)

    Ako reserve_last_fold=True, zadnji fold se cuva kao finalni temporalni test
    i ne koristi se za Optuna tuning.
    """
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col])
    d = d.sort_values(date_col).reset_index(drop=True)

    start_date = d[date_col].min()
    end_date = d[date_col].max()

    first_cutoff = start_date + pd.DateOffset(months=min_train_months)
    last_cutoff = end_date - pd.DateOffset(months=val_months)

    cutoffs = pd.date_range(
        start=first_cutoff,
        end=last_cutoff,
        freq=f"{step_months}MS",
    )

    folds = []

    for i, cutoff in enumerate(cutoffs, start=1):
        val_end = cutoff + pd.DateOffset(months=val_months)

        train_idx = d.index[d[date_col] < cutoff].to_numpy()
        val_idx = d.index[(d[date_col] >= cutoff) & (d[date_col] < val_end)].to_numpy()

        if len(train_idx) < min_train_rows or len(val_idx) < min_val_rows:
            continue

        folds.append({
            "fold": len(folds) + 1,
            "cutoff": cutoff,
            "val_end": val_end,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "is_holdout": False,
        })

    if reserve_last_fold and len(folds) >= 2:
        folds[-1]["is_holdout"] = True

    print("\nTEMPORAL FOLDS")
    print("-" * 95)
    print(
        f"{'Fold':<6} {'Train od':<12} {'Train do':<12} "
        f"{'Val od':<12} {'Val do':<12} {'Train N':>10} {'Val N':>9} {'Use':>10}"
    )
    print("-" * 95)

    for f in folds:
        train_df = d.loc[f["train_idx"]]
        val_df = d.loc[f["val_idx"]]

        use = "HOLDOUT" if f["is_holdout"] else "OPTUNA"

        print(
            f"{f['fold']:<6} "
            f"{str(train_df[date_col].min().date()):<12} "
            f"{str(train_df[date_col].max().date()):<12} "
            f"{str(val_df[date_col].min().date()):<12} "
            f"{str(val_df[date_col].max().date()):<12} "
            f"{len(train_df):>10,} {len(val_df):>9,} {use:>10}"
        )

    return d, folds


def normalize_by_group(values, group):
    values = np.asarray(values, dtype=float)
    out = np.zeros_like(values, dtype=float)

    start = 0
    for gsize in group:
        end = start + gsize
        x = values[start:end]

        x_min = np.min(x)
        x_max = np.max(x)

        if x_max > x_min:
            out[start:end] = (x - x_min) / (x_max - x_min)
        else:
            out[start:end] = 0.0

        start = end

    return out


def get_query_group(dataset):
    """
    Ranking grupa mora odgovarati jednoj ponudi/upitu.
    Ako postoji ponuda_id, to je sigurnije nego klijent_id + datum.
    """
    if "ponuda_id" in dataset.columns:
        return dataset.groupby("ponuda_id", sort=False).size().to_numpy()

    return dataset.groupby([CLIENT_COL, DATE_COL], sort=False).size().to_numpy()


def align_categorical_columns(X_train, X_val, categorical_cols):
    """
    Ne kodirati train i val kategorije odvojeno.
    Odvojeni .cat.codes moze napraviti bug:
    ista sifra u trainu i valu ne mora dobiti isti integer kod.
    """
    for col in categorical_cols:
        if col not in X_train.columns or col not in X_val.columns:
            continue

        train_vals = X_train[col].astype("object")
        val_vals = X_val[col].astype("object")

        categories = pd.Index(train_vals.dropna().unique()).union(
            pd.Index(val_vals.dropna().unique())
        )

        X_train[col] = pd.Categorical(train_vals, categories=categories).codes
        X_val[col] = pd.Categorical(val_vals, categories=categories).codes

    return X_train, X_val


def build_fold_dataset(df_sorted, fold_spec, fold_name):
    cutoff = fold_spec["cutoff"]
    val_end = fold_spec["val_end"]

    train_df = df_sorted.loc[fold_spec["train_idx"]].copy()
    val_df = df_sorted.loc[fold_spec["val_idx"]].copy()

    print(f"\n--- BUILD {fold_name} ---")
    print(
        f"Train: {train_df[DATE_COL].min().date()} -> "
        f"{train_df[DATE_COL].max().date()} ({len(train_df):,})"
    )
    print(
        f"Val:   {val_df[DATE_COL].min().date()} -> "
        f"{val_df[DATE_COL].max().date()} ({len(val_df):,})"
    )

    train_dist = train_df["sif_vrsta"].value_counts(normalize=True)
    val_dist = val_df["sif_vrsta"].value_counts(normalize=True)
    dist_diff = (train_dist - val_dist).abs().mean()

    print(f"Distribucija train/val diff: {dist_diff:.4f}")

    # Lookup tablice se grade ISKLJUCIVO na train dijelu folda.
    markov_transitions = _build_markov_with_decay(train_df)
    mt_dict = markov_transitions.set_index(["from_type", "to_type"])["prob"].to_dict()

    global_pop, reg_pop = build_popularity_features(train_df)
    seg_markov_dict = build_segment_markov(train_df)

    future_pairs_train = _build_future_pairs(train_df)
    future_pairs_val = _build_future_pairs(val_df)

    # Bitno: _build_client_history_matrix mora biti temporalno sigurna.
    # Ako funkcija koristi shift/cumsum po datumu, concat train+val je OK.
    # Ako koristi total lifetime agregate, ovo curi buducnost i mora se popraviti.
    client_hist_train = _build_client_history_matrix(train_df)
    client_hist_val = _build_client_history_matrix(
        pd.concat([train_df, val_df], ignore_index=True)
    )

    history_type_cols = [c for c in client_hist_train.columns if c != "ponuda_id"]
    features = BASE_FEATURES + history_type_cols
    training_features = TRAINING_FEATURES + history_type_cols

    # Uklanjanje zadnje polise po klijentu poslije future_pairs logike.
    train_df = train_df[train_df.groupby(CLIENT_COL).cumcount(ascending=False) != 0]
    val_df = val_df[val_df.groupby(CLIENT_COL).cumcount(ascending=False) != 0]

    if len(train_df) < 100 or len(val_df) < 20:
        raise ValueError(f"{fold_name}: premalo podataka nakon uklanjanja zadnje polise")

    gc.collect()

    sc_train, td = _build_training_chunks(
        train_df,
        future_pairs_train,
        mt_dict,
        candidates,
        client_hist_train,
        global_pop,
        reg_pop,
        seg_markov_dict,
        cluster_map,
        features,
        chunks_dir=f"chunks_train_{fold_name}",
    )

    sc_val, vd = _build_training_chunks(
        val_df,
        future_pairs_val,
        mt_dict,
        candidates,
        client_hist_val,
        global_pop,
        reg_pop,
        seg_markov_dict,
        cluster_map,
        features,
        chunks_dir=f"chunks_val_{fold_name}",
    )

    train_dataset = merge_chunks(sc_train, td)
    val_dataset = merge_chunks(sc_val, vd)

    max_pg = len(all_types) * 7

    train_dataset = cap_candidates(train_dataset, max_pg)
    val_dataset = cap_candidates(val_dataset, max_pg)

    train_dataset = train_dataset.sort_values(
        [CLIENT_COL, DATE_COL]
    ).reset_index(drop=True)

    val_dataset = val_dataset.sort_values(
        [CLIENT_COL, DATE_COL]
    ).reset_index(drop=True)

    # Negative sampling samo na trainu.
    train_dataset = sample_negatives(train_dataset)
    train_dataset = train_dataset.sort_values(
        [CLIENT_COL, DATE_COL]
    ).reset_index(drop=True)

    drop_debug_features = ["target_is_renewal", "client_cluster"]

    available_features = [
        c for c in training_features
        if c in train_dataset.columns and c in val_dataset.columns
        and c not in drop_debug_features
    ]

    X_train = train_dataset[available_features].copy()
    X_val = val_dataset[available_features].copy()

    X_train, X_val = align_categorical_columns(
        X_train,
        X_val,
        CATEGORICAL_FEATURES,
    )

    y_train = train_dataset["label"].astype(int).to_numpy()
    y_val = val_dataset["label"].astype(int).to_numpy()

    group_train = get_query_group(train_dataset)
    group_val = get_query_group(val_dataset)

    assert group_train.sum() == len(X_train)
    assert group_val.sum() == len(X_val)

    return {
        "fold": fold_spec["fold"],
        "fold_name": fold_name,
        "is_holdout": fold_spec["is_holdout"],
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
        "group_train": group_train,
        "group_val": group_val,
        "features": available_features,
    }


def suggest_lgbm_params(trial):
    return {
        "objective": "lambdarank",
        "n_estimators": trial.suggest_int("n_estimators", 300, 1800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 96),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 120),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }


def evaluate_ranker_scores(scores, y, group):
    return {
        "hr1": hit_rate_at_k(scores, y, group, k=1),
        "hr3": hit_rate_at_k(scores, y, group, k=3),
        "hr5": hit_rate_at_k(scores, y, group, k=5),
        "mrr5": mrr_at_k(scores, y, group, k=5),
    }


def optuna_objective(trial, fold_cache):
    params = suggest_lgbm_params(trial)

    fold_scores = []

    for i, fd in enumerate(fold_cache):
        model = LGBMRanker(**params)

        model.fit(
            fd["X_train"],
            fd["y_train"],
            group=fd["group_train"],
            eval_set=[(fd["X_val"], fd["y_val"])],
            eval_group=[fd["group_val"]],
            eval_at=[1, 3, 5],
            callbacks=[
                early_stopping(stopping_rounds=80, verbose=False),
            ],
        )

        preds = model.predict(fd["X_val"])

        metrics = evaluate_ranker_scores(
            preds,
            fd["y_val"],
            fd["group_val"],
        )

        # Glavna metrika. HR@3 je obicno dobar izbor za preporuku top 3.
        # MRR@5 dodaje nagradu ako je pogodak visoko rangiran.
        score = 0.75 * metrics["hr3"] + 0.25 * metrics["mrr5"]
        fold_scores.append(score)

        trial.report(float(np.mean(fold_scores)), step=i)

        if trial.should_prune():
            raise optuna.TrialPruned()

        del model
        gc.collect()

    return float(np.mean(fold_scores))


def train_and_evaluate_best(params, fd, label):
    model = LGBMRanker(**params)

    model.fit(
        fd["X_train"],
        fd["y_train"],
        group=fd["group_train"],
        eval_set=[(fd["X_val"], fd["y_val"])],
        eval_group=[fd["group_val"]],
        eval_at=[1, 3, 5],
        callbacks=[
            log_evaluation(50),
            early_stopping(stopping_rounds=100, verbose=False),
        ],
    )

    preds = model.predict(fd["X_val"])
    metrics = evaluate_ranker_scores(preds, fd["y_val"], fd["group_val"])

    print(f"\n--- {label} ---")
    print("HR@1:", round(metrics["hr1"], 4))
    print("HR@3:", round(metrics["hr3"], 4))
    print("HR@5:", round(metrics["hr5"], 4))
    print("MRR@5:", round(metrics["mrr5"], 4))

    if "markov_prob" in fd["val_dataset"].columns:
        print("\nBlend model + markov:")
        model_score_norm = normalize_by_group(preds, fd["group_val"])
        markov_score_norm = normalize_by_group(
            fd["val_dataset"]["markov_prob"].values,
            fd["group_val"],
        )

        for w_model in [0.5, 0.65, 0.8, 0.9]:
            blended = (
                w_model * model_score_norm
                + (1.0 - w_model) * markov_score_norm
            )

            bm = evaluate_ranker_scores(blended, fd["y_val"], fd["group_val"])

            print(
                f"model={w_model:.2f}, markov={1-w_model:.2f} "
                f"| HR@1={bm['hr1']:.4f} "
                f"| HR@3={bm['hr3']:.4f} "
                f"| HR@5={bm['hr5']:.4f} "
                f"| MRR@5={bm['mrr5']:.4f}"
            )

    return model, preds, metrics

In [ ]:
df_sorted, folds = make_temporal_folds(
    df,
    val_months=3,
    min_train_months=6,
    step_months=3,
    reserve_last_fold=True,
)

optuna_folds = [f for f in folds if not f["is_holdout"]]
holdout_folds = [f for f in folds if f["is_holdout"]]

fold_cache = []

for f in optuna_folds:
    fd = build_fold_dataset(
        df_sorted,
        f,
        fold_name=f"f{f['fold']}",
    )
    fold_cache.append(fd)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        multivariate=True,
        group=True,
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=1,
    ),
)

study.optimize(
    lambda trial: optuna_objective(trial, fold_cache),
    n_trials=50,
    show_progress_bar=True,
)

print("\nBEST OPTUNA SCORE:", study.best_value)
print("BEST PARAMS:")
print(study.best_params)

best_params = suggest_lgbm_params(optuna.trial.FixedTrial(study.best_params))

cv_results = []

for fd in fold_cache:
    model, preds, metrics = train_and_evaluate_best(
        best_params,
        fd,
        label=f"BEST MODEL CV Fold {fd['fold']}",
    )

    cv_results.append({
        "fold": fd["fold"],
        **metrics,
    })

cv_results = pd.DataFrame(cv_results)

print("\nCV SUMMARY")
print(cv_results.round(4))
print("\nMean:")
print(cv_results[["hr1", "hr3", "hr5", "mrr5"]].mean().round(4))
print("\nStd:")
print(cv_results[["hr1", "hr3", "hr5", "mrr5"]].std().round(4))

if holdout_folds:
    holdout_fd = build_fold_dataset(
        df_sorted,
        holdout_folds[0],
        fold_name=f"holdout_f{holdout_folds[0]['fold']}",
    )

    holdout_model, holdout_preds, holdout_metrics = train_and_evaluate_best(
        best_params,
        holdout_fd,
        label="FINAL TEMPORAL HOLDOUT",
    )